## How to run this notebook

There are two ways. Both are safe; choose either.

**Way 1: run your own copy.** No warning, and your changes are kept.

- At the top of this page, choose **Copy to Drive** (or **File → Save a copy in Drive**).
- Your copy opens in a new tab. It is saved in your Google Drive, in the folder **Colab Notebooks**.
- In your copy, choose **Runtime → Run all**.

**Way 2: run this page directly.** Quicker, but your changes are not kept.

- Choose **Runtime → Run all**.
- Colab shows this warning:

  > Warning: This notebook was not authored by Google. This notebook is being loaded from GitHub. It may request access to your data stored with Google, or read data and credentials from other sessions. Please review the source code before executing this notebook.

- Choose **Run anyway**.

**Why it is safe.** Colab shows this warning for every notebook that does not come from Google, before it has looked at the code. This notebook does not ask for your Google account, your Drive or any other data of yours. It only saves the table it analyses, and any figures it draws, in this session, which ends when you close the page. You can read every line of the code below before you run it.

# Chip packing lines: is Line B less consistent?

**Test for Homogeneity of Variance** · a worked example from the StatNex teaching guide

**The story.** A snack factory fills 150 g bags of chips on two packing lines, and 14 bags were weighed from each line. Both lines look fine on average, but the quality team suspects Line B is erratic.

**The question.** Is Line B's fill weight more variable than Line A's?

The first cells save the table in this session. The rest is the calculation StatNex runs, written out in Python. It does not use StatNex, and every number it prints was checked against the app's.

## The table

This cell saves the table `03_two_sample/homogeneity_of_variance.csv` (14 rows) as `data.csv`, the file the code reads.

In [ ]:
with open("data.csv", "w", encoding="utf-8") as file:
    file.write("""\
No.,Line A,Line B
1.0,150.3,152.2
2.0,151.4,149.5
3.0,151.0,151.3
4.0,149.2,151.4
5.0,151.2,151.2
6.0,150.9,150.2
7.0,150.7,155.4
8.0,151.4,150.2
9.0,151.2,153.6
10.0,150.4,153.7
11.0,150.6,148.2
12.0,151.3,151.6
13.0,150.8,153.3
14.0,151.9,149.6
""")

## Libraries

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

pd.set_option("display.width", 120)

## The data

In [ ]:
data = pd.read_csv("data.csv", keep_default_na=False, na_values=[""])
groups = ["Line A", "Line B"]
confidence = 95 / 100
alpha = 5 / 100

## Descriptive statistics

In [ ]:
# One column per group. Text and blank cells are skipped, so the groups may differ
# in size.
values = [pd.to_numeric(data[name], errors="coerce").dropna().to_numpy(float) for name in groups]
descriptive = pd.DataFrame({"n": [len(v) for v in values], "mean": [v.mean() for v in values],
                            "variance": [v.var(ddof=1) for v in values]}, index=groups)
descriptive["SD"] = np.sqrt(descriptive["variance"])
descriptive["SE"] = descriptive["SD"] / np.sqrt(descriptive["n"])
print(descriptive)

## Ratio of variances and its confidence interval

In [ ]:
# The larger variance goes on top, so the ratio is at least 1. If the two groups were
# equally spread, it would land near 1; the interval is the range of true ratios the
# data are consistent with. SciPy has no one-line F test for two variances, so the F
# distribution is used directly.
(n1, n2), (v1, v2) = descriptive["n"], descriptive["variance"]
if v2 > v1:
    ratio, df1, df2 = v2 / v1, n2 - 1, n1 - 1
else:
    ratio, df1, df2 = v1 / v2, n1 - 1, n2 - 1
quantile = 0.5 + confidence / 2
lower = ratio / stats.f.ppf(quantile, df1, df2)
upper = ratio * stats.f.ppf(quantile, df2, df1)
print(f"\nRatio of variances = {ratio:.6g}")
print(f"{confidence:.0%} confidence interval: {lower:.6g} to {upper:.6g}")

## F test for equal variances (two-sided)

In [ ]:
p = min(1.0, 2 * min(stats.f.sf(ratio, df1, df2), stats.f.cdf(ratio, df1, df2)))
# The F test assumes normal data in both groups, and is very sensitive to heavy
# tails: with them it reports "different spreads" far too often. Levene's test, in
# the next section, is the robust check.
# Do not use this test to choose between the pooled and Welch t tests: use Welch's.
# Power: the chance the two-sided test at alpha rejects if the true ratio equals the
# one observed. Computed from the observed ratio, it adds nothing to the P value; plan
# a study's power from a ratio that matters, chosen in advance.
upper_critical = stats.f.ppf(1 - alpha / 2, df1, df2)
lower_critical = stats.f.ppf(alpha / 2, df1, df2)
power = stats.f.sf(upper_critical / ratio, df1, df2) + stats.f.cdf(lower_critical / ratio, df1, df2)
test = pd.DataFrame([[ratio, df1, df2, p, power]], columns=["F", "df1", "df2", "P", "power"])
print()
print(test.to_string(index=False))

## Bartlett's test and Levene's test

In [ ]:
# Tests of equal variances for any number of groups. Bartlett's test compares the log
# of the pooled variance with the groups' own; like the F test it assumes normal data.
# Levene's test is a one-way ANOVA of each value's distance from the centre of its
# group: from the mean (Levene), or from the median (Brown-Forsythe), which a few
# extreme values move less. Neither squares the distances, so heavy tails mislead
# them far less than they mislead Bartlett's test: read them when the shape of the
# data is uncertain.
def levene_test(center):
    # F divides by how much the distances vary inside the groups. Where they do not
    # vary at all -- two values in every group, say -- there is no test, and SciPy
    # would print an infinite F: NaN says so instead. "Not at all" allows for rounding:
    # a sum of squares below (64 x machine epsilon)^2 of the values' own is noise.
    middle = np.median if center == "median" else np.mean
    distances = [np.abs(v - middle(v)) for v in values]
    within = sum(((d - d.mean()) ** 2).sum() for d in distances)
    if within <= (64 * np.finfo(float).eps) ** 2 * sum((v**2).sum() for v in values):
        return np.nan, np.nan
    return tuple(stats.levene(*values, center=center))


k, total = len(values), sum(len(v) for v in values)
bartlett = stats.bartlett(*values)
levene = levene_test("mean")
brown_forsythe = levene_test("median")
variance_tests = pd.DataFrame(
    [[bartlett.statistic, k - 1, np.nan, bartlett.pvalue],
     [levene[0], k - 1, total - k, levene[1]],
     [brown_forsythe[0], k - 1, total - k, brown_forsythe[1]]],
    index=["Bartlett", "Levene (mean)", "Levene (median; Brown-Forsythe)"],
    columns=["statistic", "df1", "df2", "P"])
print()
print(variance_tests)

## What the result says

Both lines fill about the same on average: 150.9 g on Line A and 151.5 g on Line B. The spread is different. The standard deviation, the typical distance of a bag from its line's average, is 0.65 g on Line A and 1.97 g on Line B, about three times as wide.

The test compares the variances (the standard deviations squared). Their ratio is the F statistic, 9.16. The P value is 0.0003: the chance of a ratio at least this far from 1 if both lines really had the same spread. The 95% confidence interval for the true ratio runs from 2.94 to 28.5. It is wide, because 14 bags per line are few, but it stays well above 1.

So yes, Line B's fill weight is more variable than Line A's.

## Your own data

Open the **Files** panel (the folder icon on the left) and upload a CSV file. Then change `"data.csv"` in the `read_csv` line to its name, change the column names in the code to yours, and choose **Run all** again.